[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/25_time_series/02_baselines_and_smoothing.ipynb)

# Bài 02 — Baseline và làm trơn hàm mũ

**🎯 Mục tiêu**
- Thiết lập bài toán dự báo: tầm dự báo $h$, chia theo thời gian; metric **MAE, RMSE, MAPE, MASE**.
- Các **baseline**: trung bình, naive, seasonal naive, drift — và vì sao chúng khó bị đánh bại.
- **Làm trơn hàm mũ** from scratch: SES (mức), Holt (mức + xu hướng), Holt–Winters (mức + xu hướng + mùa vụ); ước lượng tham số bằng tối thiểu sai số một bước.
- Khoảng dự báo và kiểm tra độ phủ.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/25_time_series

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

from ml_course.visualization import set_style

set_style()

## 1. Thiết lập: dự báo 13 tuần cuối

Dùng dữ liệu đến hết 30/9/2024 để dự báo 91 ngày tiếp theo (Q4/2024). **Không bao giờ** xáo trộn: test phải nằm **sau** train.

In [ ]:
df = pd.read_csv("../../data/sample/sales_daily.csv", parse_dates=["date"]).set_index("date").asfreq("D")
s = df["sales"]
H = 91
train, test = s.iloc[:-H], s.iloc[-H:]
print(f"train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} ngày) | test: {test.index.min().date()} → {test.index.max().date()}")

**Metric:**
- MAE $= \text{mean}|y - \hat y|$, RMSE — cùng đơn vị với dữ liệu, khó so sánh giữa các chuỗi.
- MAPE $= \text{mean}|y - \hat y|/|y|$ — theo %, nhưng bùng nổ khi $y \approx 0$ và phạt dự báo cao/thấp không đối xứng.
- **MASE** $= \text{MAE} / \text{MAE}_{\text{seasonal naive in-sample}}$ — chia cho sai số của seasonal naive (một bước, trên train). MASE < 1: tốt hơn seasonal naive. Không phụ thuộc đơn vị, dùng được khi có số 0 (Hyndman & Koehler, 2006).

In [ ]:
m = 7
mase_scale = np.mean(np.abs(train.values[m:] - train.values[:-m]))


def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    return {"MAE": mae, "RMSE": np.sqrt(np.mean((y_true - y_pred) ** 2)),
            "MAPE %": 100 * np.mean(np.abs(y_true - y_pred) / np.abs(y_true)), "MASE": mae / mase_scale}

## 2. Baseline

| Phương pháp | Dự báo $\hat y_{T+h}$ |
|---|---|
| Trung bình | $\bar y$ |
| Naive | $y_T$ (giá trị cuối) |
| Seasonal naive | $y_{T+h-m\lceil h/m\rceil}$ (cùng thứ, tuần cuối) |
| Drift | $y_T + h\,\frac{y_T - y_1}{T - 1}$ |

In [ ]:
forecasts = {
    "trung bình": np.full(H, train.mean()),
    "naive": np.full(H, train.iloc[-1]),
    "seasonal naive": np.tile(train.values[-m:], H // m + 1)[:H],
    "drift": train.iloc[-1] + (train.iloc[-1] - train.iloc[0]) / (len(train) - 1) * np.arange(1, H + 1),
}
pd.DataFrame({k: evaluate(test, v) for k, v in forecasts.items()}).T.round(3)

## 3. Làm trơn hàm mũ (exponential smoothing)

**SES** — chỉ có **mức** $\ell_t$: trung bình có trọng số giảm theo hàm mũ của quá khứ
$$\ell_t = \alpha y_t + (1 - \alpha)\ell_{t-1}, \qquad \hat y_{T+h} = \ell_T$$
$\alpha$ lớn → bám sát dữ liệu mới; $\alpha$ nhỏ → mượt, nhớ lâu.

**Holt** — thêm **xu hướng** $b_t$; **Holt–Winters** — thêm **mùa vụ** $s_t$ chu kỳ $m$ (dạng cộng tính, áp dụng trên log doanh số để thành nhân tính):
$$\begin{aligned}
\ell_t &= \alpha(y_t - s_{t-m}) + (1 - \alpha)(\ell_{t-1} + b_{t-1})\\
b_t &= \beta(\ell_t - \ell_{t-1}) + (1 - \beta)b_{t-1}\\
s_t &= \gamma(y_t - \ell_t) + (1 - \gamma)s_{t-m}\\
\hat y_{T+h} &= \ell_T + h\,b_T + s_{T+h-m\lceil h/m\rceil}
\end{aligned}$$

Một hàm chung cho cả ba (bỏ xu hướng: `beta=None`, bỏ mùa vụ: `gamma=None`):

In [ ]:
def holt_winters(y, alpha, beta=None, gamma=None, m=7, h=0):
    """Trả về (dự báo một bước trên train, dự báo h bước sau điểm cuối)."""
    y = np.asarray(y, float)
    seasonal = gamma is not None
    level = y[:m].mean() if seasonal else y[0]
    trend = ((y[m:2 * m].mean() - y[:m].mean()) / m) if beta is not None else 0.0
    season = list(y[:m] - level) if seasonal else []
    fitted = np.empty(len(y))
    for t in range(len(y)):
        s_prev = season[t - m + m] if seasonal and t < m else (season[-m] if seasonal else 0.0)
        fitted[t] = level + trend + s_prev
        new_level = alpha * (y[t] - s_prev) + (1 - alpha) * (level + trend)
        if beta is not None:
            trend = beta * (new_level - level) + (1 - beta) * trend
        if seasonal:
            season.append(gamma * (y[t] - new_level) + (1 - gamma) * s_prev)
        level = new_level
    future = np.array([level + (k + 1) * trend + (season[len(season) - m + (k % m)] if seasonal else 0.0) for k in range(h)])
    return fitted, future


def fit_smoothing(y, use_trend, use_season, m=7):
    n_par = 1 + use_trend + use_season
    def sse(p):
        a = p[0]
        b = p[1] if use_trend else None
        g = p[-1] if use_season else None
        fitted, _ = holt_winters(y, a, b, g, m)
        return np.sum((y[m:] - fitted[m:]) ** 2)                         # bỏ m điểm đầu (khởi tạo)
    res = minimize(sse, x0=[0.3] + [0.05] * (n_par - 1), bounds=[(1e-3, 0.999)] * n_par, method="L-BFGS-B")
    return res.x

### Ảnh hưởng của $\alpha$ (SES)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
window = train["2024-06-01":]
ax.plot(window, color="gray", lw=0.7, label="dữ liệu")
for a in [0.05, 0.3, 0.9]:
    fitted, _ = holt_winters(train.values, a)
    ax.plot(window.index, fitted[-len(window):], label=f"SES α = {a}")
ax.legend()
ax.set_title("α nhỏ: mượt, chậm phản ứng; α lớn: bám sát nhiễu")
plt.show()

### So sánh trên tập test

Mô hình làm trơn được fit trên **log** doanh số (mùa vụ nhân tính → cộng tính), dự báo xong lấy `exp`.

In [ ]:
log_train = np.log(train.values)
configs = {"SES": (False, False), "Holt": (True, False), "Holt-Winters (tuần)": (True, True)}
params = {}
for name, (use_t, use_s) in configs.items():
    p = fit_smoothing(log_train, use_t, use_s)
    params[name] = p
    a, b, g = p[0], (p[1] if use_t else None), (p[-1] if use_s else None)
    forecasts[name] = np.exp(holt_winters(log_train, a, b, g, m, h=H)[1])
    print(f"{name:20s} tham số = {np.round(p, 3)}")
results = pd.DataFrame({k: evaluate(test, v) for k, v in forecasts.items()}).T.round(3).sort_values("MASE")
results

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(s["2024-08-01":], color="k", lw=0.8, label="thực tế")
for name in ["seasonal naive", "Holt-Winters (tuần)", "drift"]:
    ax.plot(test.index, forecasts[name], label=name)
ax.axvline(test.index[0], color="gray", ls=":")
ax.legend()
plt.show()

- Các baseline đơn giản (trung bình, naive, drift, seasonal naive) có MASE ~0.92–0.96 — chênh nhau không nhiều.
- Holt–Winters với mùa vụ tuần tốt nhất: bắt được chu kỳ tuần và mức gần nhất.
- **SES và Holt còn tệ hơn naive.** Bốn tuần cuối tháng 9 có mức cao (~292, do mùa vụ năm đang ở đỉnh phụ và có khuyến mãi) hơn hẳn trung bình Q4 (~265): SES/Holt mang mức cao đó vào tương lai. Naive thì "may mắn": ngày cuối train là **thứ Hai** (ngày thấp trong tuần), giá trị ≈ trung bình Q4. Tham số còn được tối ưu cho sai số **một bước**, không phải tầm 91 ngày.

> ⚠️ Bài học quan trọng: xếp hạng trên **một** điểm gốc dự báo có thể do may mắn. Bài 04 đánh giá trên nhiều điểm gốc (**backtesting**).
- Không phương pháp nào ở đây biết về **mùa vụ năm** (Q4 thấp, đỉnh Tết), **khuyến mãi**, **ngày lễ** → sai số còn lớn. Bài 04 đưa các thông tin này vào làm feature.

> 📌 Luôn báo cáo baseline. Một model "AI" phức tạp có MASE 0.95 nghĩa là chỉ hơn seasonal naive 5% — có đáng chi phí?

## 4. Khoảng dự báo

Một dự báo điểm không đủ cho quyết định (nhập bao nhiêu hàng?). Với seasonal naive, nếu sai số một-mùa-vụ trên train có độ lệch chuẩn $\hat\sigma$, khoảng dự báo xấp xỉ (Hyndman & Athanasopoulos, §5.5):
$$\hat y_{T+h} \pm z\,\hat\sigma\sqrt{k + 1}, \qquad k = \lfloor (h - 1)/m\rfloor$$
— rộng dần theo số chu kỳ phía trước.

In [ ]:
resid_sn = train.values[m:] - train.values[:-m]
sigma = resid_sn.std()
k = (np.arange(1, H + 1) - 1) // m
half = norm.ppf(0.975) * sigma * np.sqrt(k + 1)
lo, hi = forecasts["seasonal naive"] - half, forecasts["seasonal naive"] + half
coverage = np.mean((test.values >= lo) & (test.values <= hi))
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test.index, test.values, color="k", lw=0.8, label="thực tế")
ax.plot(test.index, forecasts["seasonal naive"], label="seasonal naive")
ax.fill_between(test.index, lo, hi, alpha=0.2, label="khoảng 95%")
ax.legend()
ax.set_title(f"Độ phủ thực tế của khoảng 95% trên test: {coverage:.0%}")
plt.show()
print(f"σ (sai số seasonal naive trên train) = {sigma:.1f} | độ phủ thực tế của khoảng 95% = {coverage:.1%}")
print(f"độ rộng khoảng: tuần đầu ±{half[0]:.0f}, tuần cuối ±{half[-1]:.0f} (doanh số trung bình test ≈ {test.mean():.0f})")

Độ phủ 100% nghe có vẻ tốt nhưng là **hiệu chỉnh sai**: khoảng quá rộng (cuối tầm dự báo rộng hơn cả mức doanh số) → vô dụng cho lập kế hoạch tồn kho. Hai nguyên nhân: (1) $\hat\sigma$ bị thổi phồng bởi các ngày khuyến mãi/lễ (xuất hiện ở cả $y_t$ lẫn $y_{t-7}$); (2) công thức $\sqrt{k + 1}$ giả định sai số **tích luỹ** như bước ngẫu nhiên, trong khi chuỗi này dao động quanh một mùa vụ ổn định. Khoảng dự báo sai theo cả hai hướng đều có hại: quá hẹp → rủi ro thiếu hàng; quá rộng → không thông tin. Luôn **kiểm tra** độ phủ trên dữ liệu giữ lại (bài 04 dùng quantile regression).

## 🧠 Kiểm tra nhanh

1. Vì sao không được chia train/test ngẫu nhiên với chuỗi thời gian?
2. MASE = 0.8 nghĩa là gì? Vì sao MAPE nguy hiểm khi doanh số gần 0?
3. SES với α = 1 tương đương baseline nào? α → 0?
4. Holt–Winters khác Holt ở đâu? Vì sao fit trên log?
5. Vì sao khoảng dự báo rộng dần theo tầm dự báo?

## 📌 Tóm tắt

- Chia theo thời gian; MASE so với seasonal naive; luôn có baseline.
- Làm trơn hàm mũ: SES (mức), Holt (+ xu hướng), Holt–Winters (+ mùa vụ); tham số tối thiểu sai số một bước.
- Khoảng dự báo rộng dần theo tầm; kiểm tra độ phủ thực tế.

## 📚 Tài liệu tham khảo

- Hyndman & Athanasopoulos — *Forecasting: Principles and Practice* (3rd ed.), ch. 5 & 8
- Hyndman & Koehler (2006) — *Another look at measures of forecast accuracy*
- Gardner (2006) — *Exponential smoothing: The state of the art — Part II*